In [1]:
## data process
import pandas as pd
import os, re
import numpy as np
from tqdm.notebook import tqdm

In [4]:
## Microarray data
ag_file_1 = 'BRCA/unc.edu_BRCA_AgilentG4502A_07_3.geneExp.tsv' #'BRCA/unc.edu_BRCA_AgilentG4502A_07_3.geneExp.tsv'
#ag_file = 'LAML/genome.wustl.edu_LAML_HG-U133_Plus_2.geneExp.tsv'
ag_file_2 = 'LUSC/unc.edu_LUSC_AgilentG4502A_07_3.geneExp.tsv'
df_ag1 = pd.read_csv(ag_file_1,sep="\t", comment='#',index_col=0)
df_ag2 = pd.read_csv(ag_file_2,sep="\t", comment='#',index_col=0)
df_ag = pd.concat((df_ag1, df_ag2), axis=1)
## RNA-seq data
# rs_file = 'BRCA/unc.edu_BRCA_IlluminaHiSeq_RNASeq.geneExp.tsv' #'BRCA/unc.edu_BRCA_AgilentG4502A_07_3.geneExp.tsv'
rs_file_1 = 'BRCA/unc.edu_BRCA_IlluminaHiSeq_RNASeqV2.geneExp.tsv'
# rs_file = 'LAML/unc.edu_LAML_IlluminaHiSeq_RNASeqV2.geneExp.tsv'
rs_file_2 = 'LUSC/unc.edu_LUSC_IlluminaHiSeq_RNASeqV2.geneExp.tsv'
df_rs1 = pd.read_csv(rs_file_1,sep="\t", comment='#',index_col=0)
df_rs2 = pd.read_csv(rs_file_2,sep="\t", comment='#',index_col=0)
df_rs = pd.concat((df_rs1, df_rs2), axis=1)

In [202]:
rs_file1 = 'BRCA/unc.edu_BRCA_IlluminaHiSeq_RNASeq.geneExp.tsv'
df_rs1 = pd.read_csv(rs_file1,sep="\t", comment='#',index_col=0)

rs_file2 = 'BRCA/unc.edu_BRCA_IlluminaHiSeq_RNASeqV2.geneExp.tsv'
df_rs2 = pd.read_csv(rs_file2,sep="\t", comment='#',index_col=0)

print(df_rs1.shape, df_rs2.shape)

(20532, 884) (20531, 1218)


In [5]:
print(df_ag.shape, df_rs.shape)

(17814, 753) (20531, 1772)


In [12]:
df_ag2.shape

(17814, 156)

In [149]:
# df_ag.columns = [x[:16] for x in df_ag.columns ]
# df_rs.columns = [x[:16] for x in df_rs.columns ]

In [13]:
common_id = (set(df_ag.columns) & set(df_rs.columns))
print(len(common_id))

739


In [14]:
df_ag = df_ag.loc[:, list(common_id)]
df_rs = df_rs.loc[:, list(common_id)]

In [15]:
gene_ag = [x.upper() for x in df_ag.index]
df_ag.index=gene_ag

In [16]:
from collections import Counter
gene_pool = Counter(gene_ag)

In [17]:
used_ind_rs, used_ind_ag = [], []
for ind, d in df_rs.iterrows():
    tmp_gene= re.split(r'\|', ind)[0]
    if gene_pool[tmp_gene.upper()]==1:
        used_ind_rs.append(ind)
        used_ind_ag.append(tmp_gene.upper())

In [18]:
df_ag = df_ag.loc[used_ind_ag,:]
df_rs = df_rs.loc[used_ind_rs,:]

In [19]:
df_rs = df_rs.transpose()
df_ag = df_ag.transpose()

In [20]:
df_rs.columns = df_ag.columns

In [21]:
print(df_ag.shape, df_rs.shape)

(739, 16148) (739, 16148)


In [22]:
len(df_ag.index.unique())

739

In [23]:
cols_with_nan_ag = df_ag.columns[df_ag.isnull().any()].tolist()
cols_with_nan_rs = df_rs.columns[df_rs.isnull().any()].tolist()
filter_cols = list(set(cols_with_nan_ag + cols_with_nan_rs)) 

In [24]:
df_ag_new = df_ag.drop(filter_cols,axis=1)
df_rs_new = df_rs.drop(filter_cols,axis=1)

In [25]:
df_ag_new.max().max()

np.float64(11.055625)

In [26]:
df_rs_new.max().max()

np.float64(1725510.0)

In [27]:
df_rs_new = np.log2(df_rs_new + 1)

In [97]:
df_ag_new = np.log2(df_ag_new + 1)
df_rs_new = np.log2(df_rs_new + 1)

In [28]:
print(df_ag_new.shape, df_rs_new.shape)

(739, 15623) (739, 15623)


In [37]:
BRCA_samples = [x for x in df_ag_new.index if x in df_ag1.columns]
with open('BRCA_LUSC/BRCA_samples.txt','w') as f:
    for x in BRCA_samples:
        f.write(x+'\n')

In [38]:
df_ag_new.to_csv(f'BRCA_LUSC/COMB_ag.tsv',sep="\t")
df_rs_new.to_csv(f'BRCA_LUSC/COMB_rs.tsv',sep="\t")